In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Grid and Time Parameters
Nx, Ny = 100, 100       # Number of grid points in x and y directions
dx, dy = 1, 1           # Grid spacing (assumed uniform)
dt = 0.001              # Time step (must be small for explicit Euler stability)
T = 1000001             # Total number of time steps

# 2. Model Parameters 
phi_m   = 0.9           # Maximum density (area fraction)
phi0    = 0.8           # Mean density for initial phi
M0      = 5             # Mobility coefficient in the phi-equation
Pe_R    = 0.001         # Persistence parameter entering the chemical potential
kappa   = 5             # Interface smoothing parameter (related to gradient energy)
chi0    = 4           # Chemotactic coefficient (couples phi and chemoattractant gradient)
D_c     = 1             # Diffusion coefficient for the chemoattractant
k       = 0.5/kappa   # Chemoattractant uptake rate constant
S       = 0.5/kappa   # Constant supply rate of chemoattractant

# 3. Initial Conditions 
phi = phi0 + 0.02 * (np.random.rand(Nx, Ny) - 0.5)
c   = 0.02 * (np.random.rand(Nx, Ny) - 0.5)

# 4. Finite Difference Operators
def laplacian(field, dx):
    return (
        np.roll(field, -1, axis=0) + np.roll(field, 1, axis=0) +
        np.roll(field, -1, axis=1) + np.roll(field, 1, axis=1) -
        4 * field
    ) / (dx ** 2)

def gradient_x(field, dx):
    return (np.roll(field, -1, axis=0) - np.roll(field, 1, axis=0)) / (2 * dx)

def gradient_y(field, dx):
    return (np.roll(field, -1, axis=1) - np.roll(field, 1, axis=1)) / (2 * dx)

# 5. Chemical Potential Function 
def compute_mu(phi, phi_m, Pe_R, dx):
    phi_clipped = np.clip(phi, 1e-6, phi_m - 1e-6)
    term1 = np.log(phi_clipped)
    term2 = 1 - 2 * phi_clipped - 0.3 * phi_clipped ** 2
    term3 = - (4 / np.pi) * Pe_R * phi_m * (
        np.log(1 - phi_clipped / phi_m)
        - phi_clipped / phi_m
        - phi_clipped
    )
    return term1 + term2 + term3

# 6. Main Time-Stepping Loop (Explicit Euler)
for t in range(T):
    # 6.1 Compute chemical potential mu and its effective form
    mu = compute_mu(phi, phi_m, Pe_R, dx)
    lap_phi = laplacian(phi, dx)
    mu_eff = mu - kappa * lap_phi

    # 6.2 Compute spatial gradients
    grad_mu_eff_x = gradient_x(mu_eff, dx)
    grad_mu_eff_y = gradient_y(mu_eff, dx)
    grad_c_x = gradient_x(c, dx)
    grad_c_y = gradient_y(c, dx)

    # 6.3 Construct flux for phi
    flux_MIPS_x = M0 * phi * grad_mu_eff_x
    flux_MIPS_y = M0 * phi * grad_mu_eff_y
    flux_chem_x = - chi0 * phi * grad_c_x
    flux_chem_y = - chi0 * phi * grad_c_y
    flux_x = flux_MIPS_x + flux_chem_x
    flux_y = flux_MIPS_y + flux_chem_y

    # 6.4 Compute divergence of the flux
    div_flux = gradient_x(flux_x, dx) + gradient_y(flux_y, dx)

    # 6.5 Update phi using the explicit Euler method
    phi_new = phi + dt * div_flux

    # 6.6 Update chemoattractant field c
    lap_c = laplacian(c, dx)
    c_new = c + dt * (D_c * lap_c - k * c + S * phi)

    # 6.7 Synchronize updates
    phi = phi_new
    c = c_new

    # 6.8 Enforce physical bounds
    phi = np.clip(phi, 0, 1)
    c = np.clip(c, 0, None)

    # 6.9 Visualization every 100000 steps (with text in upper-left)
    if t % 10000 == 0:
        print(f"Time step: {t}, t = {t * dt:.3f}")
        plt.figure(figsize=(12, 5))

        # 6.9.1 Plot phi heatmap
        ax = plt.subplot(1, 2, 1)
        ax.set_title(f"phi at t = {t * dt:.3f}")
        im = ax.imshow(phi, origin='lower', cmap='viridis', extent=[0, Nx, 0, Ny])
        plt.colorbar(im, ax=ax, label="Density (phi)")

        # Compute instantaneous velocity v = flux / (phi + eps)
        eps = 1e-6
        v_x = flux_x / (phi + eps)
        v_y = flux_y / (phi + eps)

        # Extract high-density stripe mask
        phi_threshold = np.mean(phi)
        mask = (phi > phi_threshold)

        if np.any(mask):
            mean_vx = np.mean(v_x[mask])
            mean_vy = np.mean(v_y[mask])
        else:
            mean_vx = 0.0
            mean_vy = 0.0

        # Scale the average velocity for plotting
        factor_global = 50.0
        Uglob = mean_vx * factor_global
        Vglob = mean_vy * factor_global

        # 6.9.2 Plot c heatmap
        ax2 = plt.subplot(1, 2, 2)
        ax2.set_title(f"c at t = {t * dt:.3f}")
        im2 = ax2.imshow(c, origin='lower', cmap='inferno', extent=[0, Nx, 0, Ny])
        plt.colorbar(im2, ax=ax2, label="Chemoattractant (c)")

        plt.tight_layout()
        plt.show()